Set Evalhub environment variables

In [ ]:
EVALHUB_URL='https://evalhub-evalhub.apps.cluster-n8b82.n8b82.sandbox5200.opentlc.com'
TENANT_NS='evalhub'
TOKEN='sha256~qgp4Td0pjhZ3bpygEv2RKM3sRbKichvwsZDCaT7mICg'
!echo $EVALHUB_URL
!echo $TENANT_NS
!echo $TOKEN

In [ ]:
pip list | grep -i eval-hub

eval-hub-sdk            1.0.5
Note: you may need to restart the kernel to use updated packages.


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/asheet/Documents/projects/ai-security-pipeline/fork-ai-model-security-pipeline/.venv/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py", line 566, in _log_error
    f.result()
  File "/Users/asheet/Documents/projects/ai-security-pipeline/fork-ai-model-security-pipeline/.venv/lib/python3.11/site-packages/ipykernel/kernelbase.py", line 607, in shell_channel_thread_main
    idents, msg2 = self.session.feed_identities(msg, copy=False)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/asheet/Documents/projects/ai-security-pipeline/fork-ai-model-security-pipeline/.venv/lib/python3.11/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/asheet/Documents/proje

Test and check the evalhub url to make sure it is running and healthy

In [2]:
!curl $EVALHUB_URL/api/v1/health | jq .

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    66  100    66    0     0    512      0 --:--:-- --:--:-- --:--:--   515
{
  "status": "healthy",
  "timestamp": "2026-10-07T15:50:18.852312639Z"
}


Check to see what providers are enabled

In [ ]:
!curl -s -H "Authorization: Bearer $TOKEN" -H "X-Tenant: $TENANT_NS" $EVALHUB_URL/api/v1/evaluations/providers | jq .

In [ ]:
pip install "eval-hub-sdk[client]"

In [257]:
from evalhub import SyncEvalHubClient

client = SyncEvalHubClient(
    base_url="https://evalhub-evalhub.apps.cluster-n8b82.n8b82.sandbox5200.opentlc.com",
    tenant="evalhub",
    auth_token="sha256~qgp4Td0pjhZ3bpygEv2RKM3sRbKichvwsZDCaT7mICg"
)

In [258]:
for provider in client.providers.list():
    print(f"{provider.resource.id}: {provider.name}")

benchmarks = client.benchmarks.list(provider_id="guidellm")
for b in benchmarks:
    print(f"  {b.id}: {b.name}, pass criteria: {b.pass_criteria}, primary score: {b}")

lm_evaluation_harness: LM Evaluation Harness
guidellm: GuideLLM
garak: Garak
  sweep: Auto-discover optimal load, pass criteria: threshold=10.0, primary score: id='sweep' url=None name='Auto-discover optimal load' description='Automatically sweeps from low to high concurrency to find the best throughput-latency tradeoff.' category='performance' metrics=['requests_per_second', 'prompt_tokens_per_second', 'output_tokens_per_second', 'mean_ttft_ms', 'mean_itl_ms'] num_few_shot=0 dataset_size=0 tags=['performance', 'throughput', 'latency', 'guidellm', 'auto'] primary_score=PrimaryScore(metric='output_tokens_per_second', lower_is_better=False) pass_criteria=PassCriteria(threshold=10.0) agent=None
  throughput: Maximum throughput discovery, pass criteria: threshold=10.0, primary score: id='throughput' url=None name='Maximum throughput discovery' description="Finds the server's maximum requests-per-second by gradually increasing load until saturation." category='performance' metrics=['request

In [ ]:
from evalhub import CollectionRef
from evalhub.models import JobSubmissionRequest, ModelConfig, BenchmarkConfig, PassCriteria, PrimaryScore, ExperimentConfig, ExperimentTag

job = client.jobs.submit(JobSubmissionRequest(
    name="my-eval",
    model=ModelConfig(
        url="https://redhatai-qwen3-8b-fp8-dynamic-evalhub.apps.cluster-n8b82.n8b82.sandbox5200.opentlc.com/v1",
        name="redhatai-qwen3-8b-fp8-dynamic"
    ),
    experiment=ExperimentConfig(name="qwen3-quick-eval", tags=[ExperimentTag(key="model_name", value="qwen3-8b-fp8-dynamic")]),
    benchmarks=[
        BenchmarkConfig(provider_id="garak", id="quick",
            primary_score=PrimaryScore(metric="attack_success_rate", lower_is_better=True),
            pass_criteria=PassCriteria(threshold=0.3)),
    ],
), tenant="evalhub")

job_id = job.resource.id
print(f"Job ID: {job_id}")

In [273]:
from evalhub import CollectionRef
from evalhub.models import JobSubmissionRequest, ModelConfig, BenchmarkConfig, PassCriteria, PrimaryScore, ExperimentConfig, ExperimentTag

job = client.jobs.submit(JobSubmissionRequest(
    name="my-eval",
    model=ModelConfig(
        url="https://redhatai-qwen3-8b-fp8-dynamic-evalhub.apps.cluster-n8b82.n8b82.sandbox5200.opentlc.com/v1",
        name="redhatai-qwen3-8b-fp8-dynamic"
    ),
    experiment=ExperimentConfig(name="qwen3-quick-perf-test", tags=[ExperimentTag(key="model_name", value="qwen3-8b-fp8-dynamic")]),
    benchmarks=[
        BenchmarkConfig(provider_id="guidellm", id="quick_perf_test",
            primary_score=PrimaryScore(metric="requests_per_second", lower_is_better=False),
            pass_criteria=PassCriteria(threshold=0.5),
            parameters={"data": "prompt_tokens=256,output_tokens=64", "max_seconds": 25, "request_type": "chat_completions", "warmup": "10%"}
        )
    ],
), tenant="evalhub")

job_id = job.resource.id
print(f"Job ID: {job_id}")

Job ID: 0eb679ed-6e69-454b-b4cf-985e75545ee3


In [252]:
from evalhub import (
    JobLogOptions,
    SyncEvalHubClient,
    JobLogOptions,
)

with SyncEvalHubClient(base_url="https://evalhub-evalhub.apps.cluster-n8b82.n8b82.sandbox5200.opentlc.com",
    tenant="evalhub",
    auth_token="sha256~S1Ipoq8Q5FtbPV7yvvXBxosBYZm5JaawwyGBOLZtu1g") as client:
    for update in client.jobs.watch_logs(
        job_id,
        options=JobLogOptions(tail_lines=500),
        poll_interval=2.0,
    ):
        if update.logs:
            print(update.logs, end="")

Server error 500 for https://evalhub-evalhub.apps.cluster-n8b82.n8b82.sandbox5200.opentlc.com/api/v1/evaluations/jobs/2726ec9c-4e9f-49b0-b1a7-e360bf69ef0e/logs, retrying in 0.77s (1/3)


=== pod=2726ec9c-4e9f-49b0-b1a7-e3-a3490799-d02f-4d16-80d2-7f9a89cbzk92 container=adapter benchmark_id=quick ===
2026-10-08 17:56:43,871 - evalhub.adapter.models.adapter - INFO - Loading job spec from /meta/job.json
2026-10-08 17:56:43,872 - llama_stack_provider_trustyai_garak.evalhub.garak_adapter - INFO - Loaded job 2726ec9c-4e9f-49b0-b1a7-e360bf69ef0e
2026-10-08 17:56:43,872 - llama_stack_provider_trustyai_garak.evalhub.garak_adapter - INFO - Benchmark: quick
2026-10-08 17:56:43,873 - llama_stack_provider_trustyai_garak.evalhub.garak_adapter - INFO - Model: redhatai-qwen3-8b-fp8-dynamic
2026-10-08 17:56:43,915 - llama_stack_provider_trustyai_garak.evalhub.garak_adapter - INFO - Starting Garak job 2726ec9c-4e9f-49b0-b1a7-e360bf69ef0e for benchmark quick
2026-10-08 17:56:43,935 - httpx - INFO - HTTP Request: POST http://localhost:8080/api/v1/evaluations/jobs/2726ec9c-4e9f-49b0-b1a7-e360bf69ef0e/events "HTTP/1.1 204 No Content"
2026-10-08 17:56:43,935 - llama_stack_provider_trustyai_ga

In [267]:
from evalhub.models import JobStatus

completed_jobs = client.jobs.list(status=JobStatus.COMPLETED, limit=10)
for job in completed_jobs:
    print(f"{job.id}: {job.state}")

f0872486-18a9-410c-9909-eb8eeaccc664: JobStatus.COMPLETED
c9a9da01-f469-42c2-9931-4a43208fe6aa: JobStatus.COMPLETED
8f997510-1c5e-488e-ba4e-57f37b9cde60: JobStatus.COMPLETED
89e014c1-82ad-43d0-b413-de42301015ac: JobStatus.COMPLETED
7cb67283-16f8-4caf-8717-3e54499bfedd: JobStatus.COMPLETED
63d75154-21a2-4acf-9e8c-ddbaf15e954e: JobStatus.COMPLETED
552668c7-43ff-4571-8785-c2555807cd04: JobStatus.COMPLETED
4188ac8c-dca9-4763-80ae-cd117ca2313f: JobStatus.COMPLETED
38b24a28-dccb-48b1-982b-cf64b00aa5fc: JobStatus.COMPLETED
2726ec9c-4e9f-49b0-b1a7-e360bf69ef0e: JobStatus.COMPLETED


In [ ]:
from evalhub import SyncEvalHubClient

client = SyncEvalHubClient(
    base_url="https://evalhub-evalhub.apps.cluster-n8b82.n8b82.sandbox5200.opentlc.com",
    tenant="evalhub",
    auth_token="sha256~qgp4Td0pjhZ3bpygEv2RKM3sRbKichvwsZDCaT7mICg"
)
job = client.jobs.get("0eb679ed-6e69-454b-b4cf-985e75545ee3")
job_id = job
print(job_id.resource.id)
print(job_id.model_dump_json(indent=2))

# Access results
if job.results:
    for benchmark in job.results.benchmarks:
        print(f"Benchmark: {benchmark.provider_id}:{benchmark.id}")
else:
    print("Job results are not yet available.") 

# print(f"Checking result: {job.benchmarks[0].pass_criteria} > {job.results.benchmarks[0].metrics['attack_success_rate']}")
# if job.benchmarks[0].pass_criteria.threshold > job.results.benchmarks[0].metrics['attack_success_rate']:
#     print("Model passed the benchmark")
# else:
#     print("Model failed the benchmark")


0eb679ed-6e69-454b-b4cf-985e75545ee3
{
  "resource": {
    "id": "0eb679ed-6e69-454b-b4cf-985e75545ee3",
    "tenant": "evalhub",
    "created_at": "2026-10-09T15:18:11.299650Z",
    "updated_at": "2026-10-09T15:23:00.748296Z",
    "mlflow_experiment_id": "3",
    "message": null
  },
  "status": {
    "state": "completed",
    "message": {
      "message": "Evaluation job is completed",
      "message_code": "evaluation_job_updated",
      "message_origin": "server"
    },
    "benchmarks": [
      {
        "id": "quick_perf_test",
        "provider_id": "guidellm",
        "benchmark_index": 0,
        "status": "completed",
        "phase": null,
        "error_message": null,
        "started_at": null,
        "completed_at": "2026-10-09T15:22:59.248971Z"
      }
    ]
  },
  "results": {
    "benchmarks": [
      {
        "id": "quick_perf_test",
        "provider_id": "guidellm",
        "benchmark_index": 0,
        "metrics": {
          "mean_itl_ms": 19.026997034075208,
  

In [277]:
!curl -s -H "Authorization: Bearer $TOKEN" -H "X-Tenant: evalhub" \
    $EVALHUB_URL/api/v1/evaluations/jobs/0eb679ed-6e69-454b-b4cf-985e75545ee3 | jq .results

{
  "test": {
    "score": 0.7999399,
    "threshold": 0.5,
    "pass": true
  },
  "benchmarks": [
    {
      "id": "quick_perf_test",
      "provider_id": "guidellm",
      "benchmark_index": 0,
      "metrics": {
        "mean_itl_ms": 19.026997034075208,
        "mean_ttft_ms": 85.40800901559683,
        "output_tokens_per_second": 50.08686721904214,
        "prompt_tokens_per_second": 211.12746692229175,
        "requests_per_second": 0.7999398976024634
      },
      "artifacts": {
        "evalhub.env_card": {
          "aggregate_results": {},
          "autograder_bias": {},
          "capture_completeness": 0.15,
          "confidence_intervals": {},
          "cpu_model": "x86_64",
          "custom": {},
          "k8s_pod_labels": {},
          "k8s_resource_limits": {},
          "key_packages": {
            "torch": "2.13.0",
            "transformers": "5.14.1"
          },
          "os_info": "Linux-5.14.0-570.141.1.el9_6.x86_64-x86_64-with-glibc2.34",
          "pe